# One-DM — OPTIONAL fine-tune (Colab, GPU)

**Only run this if the pretrained One-DM fails on your handwriting.**
One-DM is one-shot by design — try `experiments/01_onedm/test_inference.py`
with the official checkpoint FIRST. 90% chance you never need this notebook.

- Repo: https://github.com/dailenson/One-DM (MIT, ECCV 2024)
- Base: official pretrained checkpoint (never train from scratch)
- Checkpoints: auto-backed-up to Google Drive, auto-resumed on reconnect

In [ ]:
# 1. Verify GPU (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

In [ ]:
MODEL_NAME = 'onedm'
REPO_URL = 'https://github.com/dailenson/One-DM'

# 2. Clone repo + install
!git clone --depth 1 $REPO_URL /content/One-DM
%cd /content/One-DM
!pip install -q -r requirements.txt gdown

In [ ]:
# ============================================================
# CHECKPOINT RESUME — survives Colab tier switches
# ------------------------------------------------------------
# Every epoch, the trainer copies its latest checkpoint to YOUR
# Google Drive. If Colab disconnects / your free tier ends:
#   1. Open THIS notebook in a new Colab session (any tier).
#   2. Run all cells top-to-bottom again.
#   3. The trainer detects the Drive checkpoint and RESUMES
#      from the last epoch instead of starting over.
# Nothing is ever stored only on Colab's throwaway disk.
# ============================================================
from google.colab import drive
import shutil, pathlib

drive.mount('/content/drive')
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/textwritter_checkpoints') / MODEL_NAME
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_CKPT = pathlib.Path('checkpoints')
LOCAL_CKPT.mkdir(exist_ok=True)

def resume_from_drive():
    """Copy newest Drive checkpoint back to local disk. Returns path or None."""
    ckpts = sorted(DRIVE_DIR.glob('*.pth'), key=lambda p: p.stat().st_mtime)
    if ckpts:
        dst = LOCAL_CKPT / ckpts[-1].name
        shutil.copy2(ckpts[-1], dst)
        print(f'[resume] found {ckpts[-1].name} on Drive -> resuming')
        return dst
    print('[resume] no Drive checkpoint -> starting from pretrained base')
    return None

def save_to_drive(src: pathlib.Path):
    shutil.copy2(src, DRIVE_DIR / src.name)
    print(f'[checkpoint] {src.name} backed up to Drive')


## 3. Download the official PRETRAINED checkpoint

Copy the Google Drive file id from the One-DM README (posted 2024-10-24)
and paste it below. This is the base we fine-tune FROM — not from scratch.

In [ ]:
PRETRAINED_GDRIVE_ID = 'PASTE_FROM_ONEDM_README'  # <-- fill this
!gdown $PRETRAINED_GDRIVE_ID -O checkpoints/one_dm_pretrained.pth

## 4. Your handwriting data

Upload a zip of cropped line images + a `labels.txt` (one line per image:
`filename.png<TAB>transcription`). 10–30 lines is plenty for adaptation.
Format matches the IAM layout the repo's dataloader expects.

In [ ]:
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()  # pick your my_handwriting.zip
DATA = pathlib.Path('/content/data')
DATA.mkdir(exist_ok=True)
for name in uploaded:
    with zipfile.ZipFile(name) as z:
        z.extractall(DATA)
!ls $DATA | head

## 5. Fine-tune (short: 2–6 h on a free T4)

The training command below follows the repo's `train.py`. Key idea:
`--resume` points at the Drive-restored checkpoint if one exists.

In [ ]:
resume_ckpt = resume_from_drive()
BASE = str(resume_ckpt or 'checkpoints/one_dm_pretrained.pth')

# Low LR = adapt to your style without forgetting general handwriting
!python train.py \
    --cfg configs/IAM64.yml \
    --pretrain $BASE \
    --lr 1e-5 \
    --batch_size 8 \
    --epochs 20 \
    --save_dir checkpoints \
    2>&1 | tail -20

# Back up whatever the trainer produced
for ckpt in sorted(pathlib.Path('checkpoints').glob('*.pth')):
    save_to_drive(ckpt)

## 6. Done — use it

Download the newest `.pth` from Drive (`MyDrive/textwritter_checkpoints/onedm/`)
and pass it to `experiments/01_onedm/test_inference.py --checkpoint ...`.

**If Colab cut you off mid-run:** reopen notebook, Run-All — it resumes
from the last Drive checkpoint automatically.